## Run reliability experiment on logistic regression

## Imports and import dataset

In [ ]:
import os
import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

drive.mount("/content/drive")

RELIABILITY_DIR = (
    "/content/drive/MyDrive/"
    "education-ml-research/ASSISTments2009/"
    "reliability_experiment/"
)

print(os.listdir(RELIABILITY_DIR))

train_df = pd.read_csv(
    os.path.join(RELIABILITY_DIR, "train.csv")
)

q1_test = pd.read_csv(
    os.path.join(RELIABILITY_DIR, "q1_test.csv")
)

q2_test = pd.read_csv(
    os.path.join(RELIABILITY_DIR, "q2_test.csv")
)

q3_test = pd.read_csv(
    os.path.join(RELIABILITY_DIR, "q3_test.csv")
)

q4_test = pd.read_csv(
    os.path.join(RELIABILITY_DIR, "q4_test.csv")
)

print("Training:", train_df.shape)
print("Q1:", q1_test.shape)
print("Q2:", q2_test.shape)
print("Q3:", q3_test.shape)
print("Q4:", q4_test.shape)

Mounted at /content/drive
['train.csv', 'q1_test.csv', 'q2_test.csv', 'q3_test.csv', 'q4_test.csv', 'sakt_reliability', 'bkt_reliability_aucs.csv', 'lr_reliability_aucs.csv', 'dkt_reliability', 'ability_stratified_auc_comparison.csv', 'ability_stratified_auc_variation.csv', 'bkt_reliability_results.csv']


/tmp/ipykernel_6475/662609171.py:22: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(


Training: (426580, 30)
Q1: (11444, 30)
Q2: (23524, 30)
Q3: (30184, 30)
Q4: (33802, 30)


/tmp/ipykernel_6475/662609171.py:38: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  q4_test = pd.read_csv(


## Prepare data

In [ ]:
def prepare_lr_data(data):
    data = data[
        ["order_id", "user_id", "skill_id", "correct"]
    ].copy()

    # Remove interactions without a skill ID
    data = data.dropna(subset=["skill_id"]).copy()

    # Convert skill ID to integer
    data["skill_id"] = (
        data["skill_id"]
        .astype(int)
    )

    # Make sure correct is numeric
    data["correct"] = pd.to_numeric(
        data["correct"],
        errors="coerce"
    )

    # Keep only valid responses
    data = data[
        data["correct"].isin([0, 1])
    ].copy()

    # Sort chronologically within each student
    data = data.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)

    return data

lr_train = prepare_lr_data(train_df)
lr_q1 = prepare_lr_data(q1_test)
lr_q2 = prepare_lr_data(q2_test)
lr_q3 = prepare_lr_data(q3_test)
lr_q4 = prepare_lr_data(q4_test)

print("Training:", lr_train.shape)
print("Q1:", lr_q1.shape)
print("Q2:", lr_q2.shape)
print("Q3:", lr_q3.shape)
print("Q4:", lr_q4.shape)

Training: (372054, 4)
Q1: (9373, 4)
Q2: (19826, 4)
Q3: (25960, 4)
Q4: (31995, 4)


## Create logistic regression features

In [ ]:
def create_lr_features(data):

    data = data.copy()


    ## Student-level history
    data["student_previous_attempts"] = (
        data.groupby("user_id").cumcount()
    )

    data["student_previous_correct"] = (
        data.groupby("user_id")["correct"]
        .cumsum()
        .shift(1)
    )

    data["student_accuracy"] = (
        data["student_previous_correct"] /
        data["student_previous_attempts"].replace(0, np.nan)
    )

    ## No previous history -> 0.5
    data["student_accuracy"] = (
        data["student_accuracy"]
        .fillna(0.5)
    )

    ## Student-skill history
    data["skill_attempts"] = (
        data.groupby(
            ["user_id", "skill_id"]
        ).cumcount()
    )

    data["skill_previous_correct"] = (
        data.groupby(
            ["user_id", "skill_id"]
        )["correct"]
        .cumsum()
        .shift(1)
    )

    data["skill_accuracy"] = (
        data["skill_previous_correct"] /
        data["skill_attempts"].replace(0, np.nan)
    )

    ## No previous history -> 0.5
    data["skill_accuracy"] = (
        data["skill_accuracy"]
        .fillna(0.5)
    )

    ## Time since previous attempt
    data["time_since_last"] = (
        data.groupby("user_id")["order_id"]
        .diff()
        .fillna(0)
    )

    return data

lr_train = create_lr_features(lr_train)
lr_q1 = create_lr_features(lr_q1)
lr_q2 = create_lr_features(lr_q2)
lr_q3 = create_lr_features(lr_q3)
lr_q4 = create_lr_features(lr_q4)

## Check features and missing values

In [ ]:
feature_columns = [
    "student_accuracy",
    "skill_accuracy",
    "skill_attempts",
    "time_since_last"
]

print(
    lr_train[
        [
            "user_id",
            "skill_id",
            "order_id",
            "student_accuracy",
            "skill_accuracy",
            "skill_attempts",
            "time_since_last",
            "correct"
        ]
    ].head(20)
)

for name, data in {
    "Training": lr_train,
    "Q1": lr_q1,
    "Q2": lr_q2,
    "Q3": lr_q3,
    "Q4": lr_q4
}.items():

    print(
        f"\n{name} missing values:"
    )

    print(
        data[feature_columns]
        .isna()
        .sum()
    )

    user_id  skill_id  order_id  student_accuracy  skill_accuracy  \
0        14         2  21617623          0.500000        0.500000   
1        14        37  21617623          0.000000        0.500000   
2        14        70  21617623          0.000000        0.500000   
3        14         2  21617632          0.000000        0.000000   
4        14        37  21617632          0.250000        1.000000   
5        14        70  21617632          0.400000        1.000000   
6        14         2  21617641          0.500000        0.500000   
7        14        37  21617641          0.428571        0.500000   
8        14        70  21617641          0.375000        0.500000   
9        14         2  21617650          0.333333        0.333333   
10       14        37  21617650          0.300000        0.333333   
11       14        70  21617650          0.272727        0.333333   
12       14         2  21617659          0.250000        0.250000   
13       14        37  21617659   

## Create and fit the model

In [ ]:
## Define training data
X_train = lr_train[feature_columns]
y_train = lr_train["correct"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

## Define logistic regression model
lr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

## Fit the model
print("Training logistic regression...")

lr_model.fit(
    X_train,
    y_train
)

print("Logistic regression training complete.")

X_train: (372054, 4)
y_train: (372054,)
Training logistic regression...
Logistic regression training complete.


## Evaluate quartile AUCs and Brier scores

In [ ]:
## Evaluate quartile AUCs and Brier scores

quartile_results = {}

for q, df in {
    "Q1": lr_q1,
    "Q2": lr_q2,
    "Q3": lr_q3,
    "Q4": lr_q4
}.items():

    X = df[feature_columns]
    y = df["correct"]

    # Predicted probability of a correct response
    predictions = lr_model.predict_proba(X)[:, 1]

    # AUC
    auc = roc_auc_score(
        y,
        predictions
    )

    # Brier score
    brier = brier_score_loss(
        y,
        predictions
    )

    quartile_results[q] = {
        "auc": auc,
        "brier_score": brier
    }

    print(
        f"{q} AUC: {auc:.6f} | "
        f"Brier Score: {brier:.6f}"
    )

Q1 AUC: 0.683918 | Brier Score: 0.207033
Q2 AUC: 0.583409 | Brier Score: 0.241635
Q3 AUC: 0.574269 | Brier Score: 0.192473
Q4 AUC: 0.692234 | Brier Score: 0.110680


## Display and save results

In [ ]:
## Display and save results

lr_results = pd.DataFrame({
    "ability_quartile": ["Q1", "Q2", "Q3", "Q4"],
    "auc": [
        quartile_results["Q1"]["auc"],
        quartile_results["Q2"]["auc"],
        quartile_results["Q3"]["auc"],
        quartile_results["Q4"]["auc"]
    ],
    "brier_score": [
        quartile_results["Q1"]["brier_score"],
        quartile_results["Q2"]["brier_score"],
        quartile_results["Q3"]["brier_score"],
        quartile_results["Q4"]["brier_score"]
    ]
})

print(lr_results)

output_path = os.path.join(
    RELIABILITY_DIR,
    "lr_reliability_results.csv"
)

lr_results.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

  ability_quartile       auc  brier_score
0               Q1  0.683918     0.207033
1               Q2  0.583409     0.241635
2               Q3  0.574269     0.192473
3               Q4  0.692234     0.110680
Saved: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/lr_reliability_results.csv


## Results

Logistic regression was trained once using the training dataset and then evaluated separately on four student ability quartiles. The model used four features: student accuracy, skill accuracy, number of previous attempts on the skill, and time since the previous attempt.

The model achieved an AUC of 0.6839 for Q1, 0.5834 for Q2, 0.5743 for Q3, and 0.6922 for Q4. Performance therefore varied considerably across ability groups, with a difference of approximately 0.118 AUC between the lowest-performing group (Q3) and highest-performing group (Q4).

The Brier scores were 0.2070 for Q1, 0.2416 for Q2, 0.1925 for Q3, and 0.1107 for Q4. The Brier score measures the accuracy of probabilistic predictions, with lower values indicating better predictive accuracy and calibration. Thus, Q4 had both the highest AUC and the lowest Brier score, indicating that logistic regression produced its strongest overall probabilistic predictions for this group. In contrast, Q2 had the highest Brier score, indicating poorer probabilistic prediction performance despite having a higher AUC than Q3.

The results also showed a non-monotonic relationship between ability and predictive performance. AUC decreased substantially from Q1 through Q3 before increasing again for Q4. The Brier scores similarly did not change consistently with ability, with Q4 showing substantially better probabilistic performance than the other groups. Therefore, higher student ability did not consistently correspond to either higher or lower predictive performance.

Overall, these findings provide preliminary evidence that student ability is associated with differences in the predictive performance of the logistic regression baseline. The variation in both AUC and Brier score across quartiles suggests that evaluating the model using only an overall AUC could conceal meaningful differences in its performance across student groups. Examining both discrimination (AUC) and probabilistic prediction quality (Brier score) provides a more complete assessment of how reliably the model performs for students of different ability levels. These results can be compared with the BKT, DKT, and SAKT experiments to determine whether different knowledge tracing approaches are affected by student ability to different degrees.
